In [23]:
from pathlib import Path
from urllib.request import urlopen
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from deepagents.backends.utils import create_file_data
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver 
from deepagents import create_deep_agent

checkpoint = MemorySaver()
backend = StateBackend

# Read skills/<name>/SKILL.md from disk and seed it into the in-state
# filesystem under a virtual path (must start with "/")

ALL_SKILLS = ["langgraph", "python", "aws", "report-writer"]

def build_skills_files(names):
    """Build the {virtual_path: file_data} dict for only the given skill
    names, so each invoke() only pays the context/token cost for the
    skills it actually needs (instead of always loading all of them)."""
    return {
        f"/skills/{name}/SKILL.md": create_file_data(
            Path(f"skills/{name}/SKILL.md").read_text(encoding="utf-8")
        )
        for name in names
    }

skills_files = build_skills_files(ALL_SKILLS)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

agent = create_deep_agent(
    model="groq:openai/gpt-oss-20b",
    backend=StateBackend(),
    skills=["/skills/"],
    checkpointer = checkpoint
)



In [19]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What skills do you have available, and when would you use each one?"
        }
    ], "files": skills_files,
},
                      
config = {"configurable": {"thread_id": "skills-demo1"}})

print(result["messages"][-1].content)

Below is a quick‑reference list of the skills I have on this machine, what each one is designed to do, and when I’d typically invoke it.

| Skill | What it does | When to use it |
|-------|--------------|----------------|
| **aws** | Handles anything that involves Amazon Web Services: writing or troubleshooting boto3 code, building CloudFormation/CDK/Terraform templates, configuring IAM, deploying Lambda/ECS/EC2/S3/Bedrock resources, or evaluating cost/security concerns. | When the user asks for AWS‑specific code snippets, architecture guidance, permission checks, or deployment scripts. |
| **langgraph** | Deals with LangGraph / LangChain‑based agent construction and debugging. Covers graph nodes/edges, state schemas, checkpointing, streaming, and the `deepagents` helpers (`create_deep_agent`, middleware, sub‑agents, etc.). | When the user wants to build, debug, or explain a LangGraph graph or a DeepAgents‑based agent. |
| **python** | General‑purpose Python help: writing, refactoring,

In [24]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How do I build a LangGraph graph with conditional routing and memory? Show a code example.",
            }
        ],
        "files": build_skills_files(["langgraph"]),  # only the skill this question needs -> saves TPM budget
    },
    config={"configurable": {"thread_id": "skills-demo2"}}
)
    
    
for message in result["messages"]:
    message.pretty_print()


================================ Human Message =================================

How do I build a LangGraph graph with conditional routing and memory? Show a code example.
================================== Ai Message ==================================
Tool Calls:
  read_file (fc_d99ff442-8438-4e20-a7de-63f42d77e9e7)
 Call ID: fc_d99ff442-8438-4e20-a7de-63f42d77e9e7
  Args:
    file_path: /skills/langgraph/SKILL.md
    limit: 1000
================================= Tool Message =================================
Name: read_file

 1  ---
 2  name: langgraph
 3  description: Use this skill whenever the user asks about building, debugging, or explaining LangGraph graphs, LangChain agents built on LangGraph, the `deepagents` library (create_deep_agent, backends, middleware, subagents), state schemas, nodes/edges, checkpointing, or streaming. Applies to work in this repo's LangChain/deepagents demos.
 4  ---
 5  
 6  # LangGraph Skill
 7  
 8  Gives the deep agent domain knowledge about Lang